In [ ]:
#pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

credit_card_data = pd.read_csv('creditcard.csv')
credit_card_data.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,0,55773,55917,51389,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,-1,140,3230,3011,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,0,59710,49986,104390,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,0,280913,283222,273160,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,-2,1512,2458,664,1814,0,0,1000,664,1500,0,0,0,0


In [18]:
# Exploratory Data Analysis (EDA)
report = sv.analyze(credit_card_data)
report.show_html('credit_card_data_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report credit_card_data_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Some Observations:

- This dataset is messy and imbalanced.
- There are many more non-default than default cases, which can lead to misleading predictions
- The labels are confusing, especially for sex, marriage, and education. They are inconsistent.
- There are different scales for the bill and payment amount columns
- The data seems to be right-skewed
- Repayment status variables directly describe the recent payment behavior.


In [19]:
# split the data into features and target variable
X = credit_card_data.drop('default payment next month', axis=1)
y = credit_card_data['default payment next month']

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y, shuffle = True)
# Train a Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train, y_train)
# Predict on the test set
y_pred_rf = rf_classifier.predict(X_test)
# Evaluate the Random Forest Classifier
print("Random Forest Classifier Report:")
print(classification_report(y_test, y_pred_rf))
print("Random Forest Classifier Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest Classifier ROC AUC Score:", roc_auc_score(y_test, rf_classifier.predict_proba(X_test)[:, 1]))

# Train an XGBoost Classifier
xgb_classifier = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_classifier.fit(X_train, y_train)
# Predict on the test set
y_pred_xgb = xgb_classifier.predict(X_test)
# Evaluate the XGBoost Classifier
print("XGBoost Classifier Report:")
print(classification_report(y_test, y_pred_xgb))
print("XGBoost Classifier Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost Classifier ROC AUC Score:", roc_auc_score(y_test, xgb_classifier.predict_proba(X_test)[:, 1]))

Random Forest Classifier Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.62      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Random Forest Classifier Accuracy: 0.8114583333333333
Random Forest Classifier ROC AUC Score: 0.7606104758075811
XGBoost Classifier Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

XGBoost Classifier Accuracy: 0.8102083333333333
XGBoost Classifier ROC AUC Score: 0.7602205274077298


In [20]:
# now with cross validation
rf_cv_scores = cross_val_score(rf_classifier, X, y, cv=5, scoring='roc_auc')
xgb_cv_scores = cross_val_score(xgb_classifier, X, y, cv=5, scoring='roc_auc')

print("Random Forest Classifier Cross-Validation ROC AUC Scores:", rf_cv_scores)
print("XGBoost Classifier Cross-Validation ROC AUC Scores:", xgb_cv_scores)

print("Random Forest Classifier Mean Cross-Validation ROC AUC Score:", np.mean(rf_cv_scores))
print("XGBoost Classifier Mean Cross-Validation ROC AUC Score:", np.mean(xgb_cv_scores))

# standard deviation of the cross validation scores
print("Random Forest Classifier Standard Deviation of Cross-Validation ROC AUC Scores:", np.std(rf_cv_scores))
print("XGBoost Classifier Standard Deviation of Cross-Validation ROC AUC Scores:", np.std(xgb_cv_scores))

Random Forest Classifier Cross-Validation ROC AUC Scores: [0.75666328 0.75822607 0.77178925 0.75510497 0.76113431]
XGBoost Classifier Cross-Validation ROC AUC Scores: [0.75875588 0.75578512 0.77587086 0.75915497 0.76646487]
Random Forest Classifier Mean Cross-Validation ROC AUC Score: 0.7605835768039675
XGBoost Classifier Mean Cross-Validation ROC AUC Score: 0.7632063375924782
Random Forest Classifier Standard Deviation of Cross-Validation ROC AUC Scores: 0.0059466301291866384
XGBoost Classifier Standard Deviation of Cross-Validation ROC AUC Scores: 0.00724296672885756


Both models performed better with cross-validation. The mean metric score tells you about the average performance across the 5-folds. The standard deviation explains how the performance changes from fold to fold.

A higher mean has a better average performance and a lower standard deviation is preferred because it indicates that a model is more stable and consistent across different train and validation splits.

In this case, the Random Forest Classifier would be better here.

In [21]:
# look at the classification report for the two models
from sklearn.metrics import confusion_matrix

print("Random Forest Classifier Classification Report:")
print(classification_report(y_test, y_pred_rf))
print('Confusion Matrix\n', confusion_matrix(y_test, y_pred_rf))

print("XGBoost Classifier Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print('Confusion Matrix\n', confusion_matrix(y_test, y_pred_xgb))

Random Forest Classifier Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.62      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Confusion Matrix
 [[3496  242]
 [ 663  399]]
XGBoost Classifier Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.93      0.88      3738
           1       0.61      0.38      0.47      1062

    accuracy                           0.81      4800
   macro avg       0.73      0.66      0.68      4800
weighted avg       0.79      0.81      0.79      4800

Confusion Matrix
 [[3483  255]
 [ 656  406]]


It does change the evaluation because it breaks down performance by class. This means that it should predict the majority class well. When recall is low, the model is missing many true defaults and vice versa.

The classification report considers other features like precision and recall. This is important and may even be preferred over a model with a higher accuracy or ROC-AUC score. You want a model that is useful, rather than just a model that performs well.

In [22]:
# using cross-validation again but also using model features to adjust for class imbalance
rf_classifier_balanced = RandomForestClassifier(random_state=42, class_weight='balanced')
xgb_classifier_balanced = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', scale_pos_weight= (y == 0).sum() / (y == 1).sum())

rf_cv_scores_balanced = cross_val_score(rf_classifier_balanced, X, y, cv=5, scoring='roc_auc')
xgb_cv_scores_balanced = cross_val_score(xgb_classifier_balanced, X, y, cv=5, scoring='roc_auc')

print("Random Forest Classifier with Balanced Class Weights Cross-Validation ROC AUC Scores:", rf_cv_scores_balanced)
print("XGBoost Classifier with Scale Pos Weight Cross-Validation ROC AUC Scores:", xgb_cv_scores_balanced)
print("Random Forest Classifier with Balanced Class Weights Mean Cross-Validation ROC AUC Score:", np.mean(rf_cv_scores_balanced))
print("XGBoost Classifier with Scale Pos Weight Mean Cross-Validation ROC AUC Score:", np.mean(xgb_cv_scores_balanced))
print("Random Forest Classifier with Balanced Class Weights Standard Deviation of Cross-Validation ROC AUC Scores:", np.std(rf_cv_scores_balanced))
print("XGBoost Classifier with Scale Pos Weight Standard Deviation of Cross-Validation ROC AUC Scores:", np.std(xgb_cv_scores_balanced))

Random Forest Classifier with Balanced Class Weights Cross-Validation ROC AUC Scores: [0.75295665 0.76093934 0.76843098 0.7562854  0.77051121]
XGBoost Classifier with Scale Pos Weight Cross-Validation ROC AUC Scores: [0.7560491  0.75647231 0.76012165 0.75726745 0.76799481]
Random Forest Classifier with Balanced Class Weights Mean Cross-Validation ROC AUC Score: 0.7618247149166104
XGBoost Classifier with Scale Pos Weight Mean Cross-Validation ROC AUC Score: 0.7595810646012292
Random Forest Classifier with Balanced Class Weights Standard Deviation of Cross-Validation ROC AUC Scores: 0.006770650123226474
XGBoost Classifier with Scale Pos Weight Standard Deviation of Cross-Validation ROC AUC Scores: 0.0044401638238332885


This impacts model performance because there are tradeoffs. You could see lower or similar accuracy when comparing different models. In addition, recall can increase and there are changes in precision due to how many reported positive cases the model predicts.

It is important to note that an imbalance adjustment is helpful when a model can make small improvements and still be precise. In this case, now the XGBoost model is better than the Random Forest classifier.

In [23]:
# XGBoost with subsample = 0.8
xgb_classifier_subsample = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', scale_pos_weight= (y == 0).sum() / (y == 1).sum(), subsample=0.8)
xgb_cv_scores_subsample = cross_val_score(xgb_classifier_subsample, X, y, cv=5, scoring='roc_auc')
print("XGBoost Classifier with Subsample Cross-Validation ROC AUC Scores:", xgb_cv_scores_subsample)
print("XGBoost Classifier with Subsample Mean Cross-Validation ROC AUC Score:", np.mean(xgb_cv_scores_subsample))
print("XGBoost Classifier with Subsample Standard Deviation of Cross-Validation ROC AUC Scores:", np.std(xgb_cv_scores_subsample))

XGBoost Classifier with Subsample Cross-Validation ROC AUC Scores: [0.75145542 0.75182203 0.7661326  0.7466013  0.75157642]
XGBoost Classifier with Subsample Mean Cross-Validation ROC AUC Score: 0.753517553529661
XGBoost Classifier with Subsample Standard Deviation of Cross-Validation ROC AUC Scores: 0.006601049063252035


By setting the subsample = 0.8, it trains each boosting round on a random 80% sample of the training data. This is helpful in reducing overfitting and improving generalization.

The subsample helped because the ROC-AUC and F1 slightly improved. In addition, the standard deviation got smaller.

In [24]:
# random forest tuning with grid search
rf_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None, 12],
    'min_samples_leaf': [1, 3]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    param_grid=rf_param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=1
)
rf_grid.fit(X, y)

print('RF best params:', rf_grid.best_params_)
print('RF best CV ROC-AUC:', rf_grid.best_score_)

RF best params: {'max_depth': 12, 'min_samples_leaf': 3, 'n_estimators': 300}
RF best CV ROC-AUC: 0.7774166551402053


In [25]:
# tuning for xgboost with scale_pos_weight and subsample
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1,
        scale_pos_weight= (y == 0).sum() / (y == 1).sum(),
        subsample=0.8
    ),
    param_grid=xgb_param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=1
)
xgb_grid.fit(X, y)

print('XGB best params:', xgb_grid.best_params_)
print('XGB best CV ROC-AUC:', xgb_grid.best_score_)

XGB best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
XGB best CV ROC-AUC: 0.7826366770385625


For the random forest, I tuned the number of trees, tree depth, and minimum leaf size.

For the XGBoost, I tuned the number of trees, tree depth, and learning rate.

I thought these were good parameters to modify because they affect the complexity and generalization of the model. 

The scores didn't improve much. They mostly stayed the same for both models. I would have to do more tuning to see if there could be any real improvement.

The XGBoost model was better out-of-the-box. It had the higher mean cross-validated ROC-AUC (0.783) compared with the Random Forest classifier model (0.777). Its fold-to-fold variability was slightly lower. The classification report did change this conclusion because it increased the recall/precision for the default class.